# Tutorial 05: Potencial de Yukawa en 2D, Observables y Almacenamiento HDF5

El **potencial de Yukawa** modela interacciones apantalladas o de corto alcance (como potenciales nucleares mediados por mesones o potenciales de Coulomb apantallados tipo Debye-Thomas-Fermi):

$$V(r) = -V_0 \frac{e^{-a r}}{r}$$

donde $V_0$ es la intensidad de la interacción (`strength`) y $a$ es el inverso del rango de apantallamiento (`range`).

### Objetivos de aprendizaje
1. Configurar una simulación 2D bajo `YukawaPotential(strength, range)`.
2. Almacenar instantáneas del campo espinorial $\psi(\mathbf{r}, t)$ y el historial completo de observables en un archivo binario jerárquico mediante `HDF5Storage`.
3. Visualizar la evolución temporal de $\langle \mathbf{r} \rangle(t)$, $\langle \mathbf{p} \rangle(t)$ y $\langle H \rangle(t)$.
4. Representar el mapa 2D de densidad de probabilidad $\rho(x,y)$ junto con el campo vectorial de corriente $\mathbf{J}(x,y)$ e inspeccionar el archivo HDF5 resultante.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Asegurar que dirac_solver sea localizable desde examples/, examples/es/ o examples/en/
for _rel in ['..', '../..']:
    _abs = os.path.abspath(_rel)
    if os.path.exists(os.path.join(_abs, 'dirac_solver')) and _abs not in sys.path:
        sys.path.insert(0, _abs)

from dirac_solver import (
    Grid,
    ConstantSpinor,
    GaussianPacket,
    PlaneWave,
    DiracProblemBuilder,
    DiracSolver,
    FreeParticle,
    ScalarPotential,
    CoulombPotential,
    YukawaPotential,
    InfiniteWellPotential,
    Observables,
    HDF5Storage,
    _core,
    electron_mass,
    c,
    hbar,
)
from dirac_solver.storage import create_animation, create_animation_1d

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12})
import h5py

---
## 1. Configuración de la Simulación 2D con `YukawaPotential`

In [ ]:
shape = (80, 80)
spacing = (0.12, 0.12)
grid = Grid(shape, spacing)
time_step = 0.004
total_time = 2.0

# Estado inicial: paquete gaussiano con momento en x e y
constant_spinor = ConstantSpinor(momentum=[4.0, 1.0, 0.0])
initial_state = GaussianPacket(constant_spinor, center=[-2.0, -1.5, 0.0], spatial_width=0.9)

# Potencial de Yukawa: V(r) = -V0 * exp(-a * r) / r
potential = YukawaPotential(strength=5.0, range=2.0)

# Condiciones de frontera periódicas
boundary_condition = _core.PeriodicBoundary()

problem = (
    DiracProblemBuilder()
    .set_grid(grid)
    .set_initial_state(initial_state)
    .set_potential(potential)
    .set_boundary_condition(boundary_condition)
    .set_time_parameters(time_step, total_time)
    .build()
)

solver = DiracSolver(problem)

---
## 2. Ejecución con `HDF5Storage` y `Observables`

Al pasar `storage_handler` y `observables_handler` a `solver.run_simulation()`, el solver guarda periódicamente el espinor $\psi$ cada `save_every_n_steps` pasos y al finalizar vuelca todas las series temporales de observables dentro del grupo `/observables` del archivo HDF5.

In [ ]:
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
hdf5_path = os.path.join(output_dir, "yukawa_2d_tutorial.hdf5")

storage = HDF5Storage(hdf5_path, problem)
observables = Observables(solver)

solver.run_simulation(
    storage_handler=storage,
    observables_handler=observables,
    save_every_n_steps=20,
)

---
## 3. Evolución Temporal de los Valores Esperados

In [ ]:
time_arr = observables.get_history('time')
expected_pos = observables.get_history('expected_position')
expected_mom = observables.get_history('expected_momentum')
expected_energy = observables.get_history('expected_energy')

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

axes[0].plot(time_arr, expected_pos[:, 0], 'b-', lw=2, label=r'$\langle x \rangle$')
axes[0].plot(time_arr, expected_pos[:, 1], 'g-', lw=2, label=r'$\langle y \rangle$')
axes[0].set_ylabel('Posición Esperada')
axes[0].set_title('Evolución de la Posición Esperada en el Potencial de Yukawa')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_arr, expected_mom[:, 0], 'b--', lw=2, label=r'$\langle p_x \rangle$')
axes[1].plot(time_arr, expected_mom[:, 1], 'g--', lw=2, label=r'$\langle p_y \rangle$')
axes[1].set_ylabel('Momento Esperado')
axes[1].set_title('Evolución del Momento Esperado')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(time_arr, expected_energy, 'crimson', lw=2)
axes[2].set_ylabel(r'Energía Esperada $\langle H \rangle$')
axes[2].set_xlabel('Tiempo t')
axes[2].set_title('Evolución de la Energía Esperada')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Visualización 2D: Densidad $\rho(x,y)$ y Corriente $\mathbf{J}(x,y)$ + Lectura de HDF5

In [ ]:
last_prob_density = observables.get_history('probability_density')[-1]
last_prob_current = observables.get_history('probability_current')[-1]

rho_grid = last_prob_density.reshape(shape)
jx_grid = last_prob_current[:, 0].reshape(shape)
jy_grid = last_prob_current[:, 1].reshape(shape)

x = np.linspace(grid.origin[0], grid.origin[0] + (shape[0] - 1) * spacing[0], shape[0])
y = np.linspace(grid.origin[1], grid.origin[1] + (shape[1] - 1) * spacing[1], shape[1])
X, Y = np.meshgrid(x, y, indexing='ij')

plt.figure(figsize=(9, 7))
plt.imshow(rho_grid.T, origin='lower', extent=[x[0], x[-1], y[0], y[-1]], cmap='viridis', alpha=0.9)
plt.colorbar(label=r'Densidad de Probabilidad $\rho(x,y)$')

skip = 4
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip],
           jx_grid[::skip, ::skip], jy_grid[::skip, ::skip],
           color='white', pivot='mid')

plt.title(f'Densidad y Corriente de Probabilidad en t = {total_time}')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()

# Inspección de la estructura del archivo HDF5 guardado
with h5py.File(hdf5_path, 'r') as f:
    print("Atributos de raíz HDF5:", dict(f.attrs))
    print("Dataset psi:", f['psi'].shape, f['psi'].dtype)
    print("Dataset time:", f['time'].shape)
    print("Observables almacenados:", list(f['observables'].keys()))